# Tutorial 9: Running on Real Quantum Hardware

This notebook demonstrates how to run qufin algorithms on IBM Quantum hardware
using the IBM Runtime backend.

**Prerequisites**: An IBM Quantum account and API token. Set via environment variable
`IBM_QUANTUM_TOKEN` or pass directly.

**Note**: This notebook uses `pip install qufin[ibm]`.

In [ ]:
import numpy as np
import os
np.random.seed(42)

## 1. IBM Runtime Backend Setup

The `IBMRuntimeBackend` wraps IBM's Sampler and Estimator primitives.

In [ ]:
# NOTE: This cell requires a valid IBM Quantum token.
# Uncomment and set your token to run on real hardware.

# from qufin.backends.ibm_runtime import IBMRuntimeBackend
#
# backend = IBMRuntimeBackend(
#     instance="ibm-q/open/main",
#     backend_name="ibm_brisbane",  # 127-qubit Eagle r3
#     token=os.environ.get("IBM_QUANTUM_TOKEN"),
# )
# print(f"Connected to: {backend.backend_id}")

# For this tutorial, we simulate with a noisy backend (shots passed at run()).
from qufin.backends.noise_models import NoisyAerBackend, IBM_EAGLE_R3

backend = NoisyAerBackend(profile=IBM_EAGLE_R3, seed=42)
print(f"Using noisy simulator: {backend.backend_id}")

## 2. Backend Auto-Selection

qufin can automatically select the best available backend.

In [ ]:
from qufin.backends.auto_select import auto_select_backend
from qiskit.circuit import QuantumCircuit

# Build a small test circuit
test_qc = QuantumCircuit(3, 3)
test_qc.h(0)
test_qc.cx(0, 1)
test_qc.cx(1, 2)
test_qc.measure([0, 1, 2], [0, 1, 2])

auto_backend = auto_select_backend(test_qc)
print(f"Auto-selected: {auto_backend.backend_id}")

## 3. Portfolio Optimization on Noisy Hardware

In [ ]:
from qufin.portfolio.qubo import PortfolioQUBO
from qufin.portfolio.optimizers.qaoa import QAOAPortfolio, QAOAConfig

mu = np.array([0.12, 0.10, 0.07, 0.03, 0.15])
cov = np.diag([0.04, 0.03, 0.02, 0.01, 0.05])

qubo = PortfolioQUBO(mu=mu, cov=cov, gamma=0.5, cardinality=2)

config = QAOAConfig(
    p=1,  # Shallow circuit for noisy hardware
    mixer="xy_ring",
    cardinality=2,
    shots=4096,
    seed=42,
)

solver = QAOAPortfolio(qubo, config, backend)
result = solver.run()

print(f"Noisy QAOA result:")
print(f"  Best bitstring: {result.best_bitstring}")
print(f"  Objective:      {result.best_objective:.6f}")
print(f"  Feasible:       {result.feasible}")

## 4. With Error Mitigation

In [ ]:
from qufin.backends.error_mitigation import calibrate_readout

# Calibrate readout for the qubits used (signature: calibrate_readout(n_qubits, backend, shots)).
cal_matrix = calibrate_readout(qubo.n_qubits, backend, shots=4096)
print(f"Readout calibration complete.")
print(f"Diagonal accuracies: {np.diag(cal_matrix)[:5].round(4)}")

## 5. Finance-Optimized Transpilation

qufin's transpiler exploits QUBO structure for circuit optimization.

In [ ]:
from qufin.backends.transpiler import FinanceTranspiler

# Build a raw QAOA-style circuit.
from qiskit.circuit import QuantumCircuit
raw_qc = QuantumCircuit(5, 5)
raw_qc.h(range(5))
for i in range(4):
    raw_qc.cx(i, i + 1)
    raw_qc.rz(0.5, i + 1)
    raw_qc.cx(i, i + 1)
raw_qc.measure(range(5), range(5))

transpiler = FinanceTranspiler(optimization_level=3, seed=42)
optimized, report = transpiler.reduce_cnot_count(raw_qc)

print(f"Original depth:  {report.original_depth}")
print(f"Optimized depth: {report.optimized_depth}")
print(f"Original CNOTs:  {report.original_cx_count}")
print(f"Optimized CNOTs: {report.optimized_cx_count}")
# Note: CNOT reduction depends on circuit structure; dense QAOA cost layers
# (cx-rz-cx with distinct controls) often have little redundancy to cancel.

## 6. Reproducibility Manifest

Every hardware run should record a reproducibility manifest.

In [ ]:
from qufin.benchmarks.manifest import build_manifest

manifest = build_manifest(
    problem_ids=["portfolio_5_k2"],
    solver_names=["qaoa-p1-xy_ring"],
    seeds=[42],
)

print("Reproducibility Manifest:")
for key, value in manifest.to_dict().items():
    print(f"  {key}: {value}")

## Summary

In this tutorial we covered:
- IBM Runtime backend setup (token, instance, backend selection)
- Automatic backend selection
- Running QAOA on noisy hardware
- Readout calibration for error mitigation
- Finance-optimized transpilation
- Reproducibility manifests

**Next**: Tutorial 10 provides an honest analysis of quantum advantage prospects.